# SAM 후보 분자 하나 추가 계산 (Colab)

맨 위 입력칸에 SMILES·이름·용매를 넣고 **런타임 → 모두 실행**을 누르면 설치 → 계산(RDKit + GFN2-xTB) → 결과 표시까지 진행됩니다.

- 코드는 GitHub 저장소 `wj7294-blip/Calculation` 의 `sam_descriptors/` 를 clone 해서 그대로 씁니다.
- 결과 데이터베이스 `database.csv` 는 Google Drive(`MyDrive/sam_descriptors/`)에 저장되어 세션이 끝나도 남습니다.
- GPU 불필요. 분자 하나에 보통 10초~2분(CPU) 걸립니다.

In [ ]:
#@title 1. 입력 { run: "auto" }
SMILES = "FC(F)(F)c1ccc(S)cc1"  #@param {type:"string"}
NAME = "4-CF3-thiophenol"  #@param {type:"string"}
SOLVENT = "auto"  #@param ["auto", "thf", "water", "gas"]
N_CONFORMERS = 20  #@param {type:"integer"}
FORCE = False  #@param {type:"boolean"}
REPO = "https://github.com/wj7294-blip/Calculation"
DRIVE_DIR = "/content/drive/MyDrive/sam_descriptors"

In [ ]:
#@title 2. 설치 (rdkit, ase, tblite, xtb 바이너리 …) — 처음 한 번만 2~3분
import os, subprocess, sys
!pip -q install rdkit ase tblite pandas numpy scipy matplotlib koreanize-matplotlib py3Dmol pyscf
if not os.path.exists("/opt/xtb/bin/xtb"):
    !wget -q https://github.com/grimme-lab/xtb/releases/download/v6.7.1/xtb-6.7.1-linux-x86_64.tar.xz -O /tmp/xtb.tar.xz
    !mkdir -p /opt/xtb && tar xf /tmp/xtb.tar.xz -C /tmp && cp -r /tmp/xtb-dist/* /opt/xtb/
os.environ["PATH"] = "/opt/xtb/bin:" + os.environ["PATH"]
os.environ["XTBPATH"] = "/opt/xtb/share/xtb"
os.environ["OMP_NUM_THREADS"] = "2"
!xtb --version | grep version

In [ ]:
#@title 3. 코드 받기 + Google Drive 연결
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)
if not os.path.exists("/content/Calculation"):
    !git clone -q {REPO} /content/Calculation
else:
    !cd /content/Calculation && git pull -q
%cd /content/Calculation/sam_descriptors
# Drive 에 저장된 database.csv 가 있으면 가져온다 (없으면 새로 만든다)
os.makedirs("results", exist_ok=True)
if os.path.exists(f"{DRIVE_DIR}/database.csv"):
    !cp "{DRIVE_DIR}/database.csv" results/database.csv
    print("Drive 의 database.csv 를 불러왔습니다.")
if os.path.isdir(f"{DRIVE_DIR}/molecules"):
    !cp -rn "{DRIVE_DIR}/molecules" results/

In [ ]:
#@title 4. 계산 실행
import importlib, add_molecule
importlib.reload(add_molecule)
solvent = None if SOLVENT == "auto" else SOLVENT
result = add_molecule.add(SMILES, NAME, solvent=solvent, nconf=N_CONFORMERS, force=FORCE, do_refresh=True)

In [ ]:
#@title 5. 결과 표시 (표 + 3D 구조 + HOMO–쌍극자 지도)
import pandas as pd, py3Dmol
from IPython.display import Image, display
db = pd.read_csv("results/database.csv")
display(db[["name","smiles","solvent","HOMO","LUMO","gap","IP","EA","dipole","dipole_axial","q_anchor_atom","q_terminal_group","TPSA","logP"]].round(3))
xyz = open(f"xyz/{result['name']}.xyz").read()
v = py3Dmol.view(width=500, height=350); v.addModel(xyz, "xyz"); v.setStyle({"stick":{}, "sphere":{"scale":0.25}}); v.zoomTo(); v.show()
display(Image("figures/map_homo_vs_dipole.png", width=900))

In [ ]:
#@title 6. Drive 에 저장 (database.csv, 분자 JSON, xyz, viewer.html)
!cp results/database.csv "{DRIVE_DIR}/database.csv"
!mkdir -p "{DRIVE_DIR}/molecules" "{DRIVE_DIR}/xyz"
!cp results/molecules/"{result['name']}".json "{DRIVE_DIR}/molecules/"
!cp xyz/"{result['name']}".xyz "{DRIVE_DIR}/xyz/"
!cp viewer.html figures/map_homo_vs_dipole.png "{DRIVE_DIR}/"
print("저장 완료:", DRIVE_DIR)

## 참고
- 같은 정규화 SMILES 가 이미 `database.csv`(또는 저장소의 `results/summary.csv`)에 있으면 재계산하지 않고 저장값을 보여 줍니다. 다시 계산하려면 `FORCE` 를 켜세요.
- 앵커/말단 원자는 자동 인식(티올 S > 포스폰산 P > 카복실 C; 말단 = 앵커에서 가장 먼 작용기)입니다. 직접 지정하려면 SMILES 에 atom-map 을 붙이세요: 앵커 `[SH:1]`, 말단 `[C:2]`, N⁺ `[N+:4]`.
- `viewer.html` 은 Drive 에서 내려받아 브라우저로 열면 됩니다(파일 하나로 동작).